# Kaggle CPU Model Export: Qwen2.5-1.5B-Instruct + RebuiltV4 Checkpoint-200

**Autonomous API Penetration Testing Agent — Standalone CPU Deployment**

Executes full LoRA adapter merging, GGUF conversion, Q4_K_M quantization, and verification.
- Base Model: `Qwen/Qwen2.5-1.5B-Instruct`
- Adapter Checkpoint: `RebuiltV4 Checkpoint-200`
- Target Deployment Format: `Q4_K_M` GGUF (~950 MB) for 8 GB RAM Windows systems.


In [ ]:
# Cell 1: Diagnostics and Provenance Verification
import os
import sys
import torch
import json

print('=' * 80)
print('STEP 1: GPU AND SOURCE ASSET VERIFICATION')
print('=' * 80)

cuda_available = torch.cuda.is_available()
print(f'[*] CUDA Available: {cuda_available}')
if cuda_available:
    print(f'[*] GPU Device Name: {torch.cuda.get_device_name(0)}')
    print(f'[*] Available VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)} GB')

# Find RebuiltV4 Checkpoint-200
candidates = [
    '/kaggle/input/rebuiltv4-training-checkpoints/checkpoint-200',
    '/kaggle/input/rebuiltv4-adapters/checkpoint-200',
    '/kaggle/input/api-security-recovery/staging_rebuiltv4_training_checkpoints/checkpoint-200',
]
adapter_dir = None
for cand in candidates:
    if os.path.exists(cand) and os.path.exists(os.path.join(cand, 'adapter_config.json')):
        adapter_dir = cand
        break

if not adapter_dir:
    # Check all subdirectories of /kaggle/input
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'adapter_config.json' in files and 'checkpoint-200' in root:
            adapter_dir = root
            break

assert adapter_dir is not None, 'Could not find RebuiltV4 Checkpoint-200 adapter directory!'
print(f'[+] Verified RebuiltV4 Checkpoint-200 at: {adapter_dir}')

# Verify files
adapter_config_path = os.path.join(adapter_dir, 'adapter_config.json')
adapter_model_path = os.path.join(adapter_dir, 'adapter_model.safetensors')
with open(adapter_config_path, 'r') as f:
    cfg = json.load(f)
print(f'[*] LoRA Rank (r): {cfg.get("r")}, Alpha: {cfg.get("lora_alpha")}, Base Model: {cfg.get("base_model_name_or_path")}')
safetensor_size = round(os.path.getsize(adapter_model_path) / (1024 * 1024), 2)
print(f'[+] Adapter safetensors size: {safetensor_size} MB')


In [ ]:
# Cell 2: Setup llama.cpp build and dependencies
import subprocess

print('=' * 80)
print('STEP 2: LLAMA.CPP TOOLCHAIN SETUP')
print('=' * 80)

# Remove incompatible torchao package to prevent PEFT dispatch error
print('[*] Cleaning environment and installing conversion dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'transformers>=4.45.0,<4.49.0', 'peft>=0.14.0', 'accelerate>=0.34.0', 'gguf>=0.10.0', 'sentencepiece'], check=True)

LLAMA_CPP_DIR = '/tmp/llama.cpp'
if not os.path.exists(LLAMA_CPP_DIR):
    print('[*] Cloning llama.cpp repository...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ggerganov/llama.cpp.git', LLAMA_CPP_DIR], check=True)

# Build llama-quantize and llama-cli with CMake
build_dir = os.path.join(LLAMA_CPP_DIR, 'build')
print('[*] Compiling llama-quantize and llama-cli...')
subprocess.run(['cmake', '-B', build_dir, LLAMA_CPP_DIR, '-DGGML_CUDA=OFF'], check=True)
subprocess.run(['cmake', '--build', build_dir, '--target', 'llama-quantize', 'llama-cli', '-j4'], check=True)

quant_bin = os.path.join(build_dir, 'bin', 'llama-quantize')
cli_bin = os.path.join(build_dir, 'bin', 'llama-cli')
assert os.path.exists(quant_bin), f'llama-quantize not found at {quant_bin}'
assert os.path.exists(cli_bin), f'llama-cli not found at {cli_bin}'
print(f'[+] llama-quantize built: {quant_bin}')
print(f'[+] llama-cli built: {cli_bin}')


In [ ]:
# Cell 3: Load and Merge LoRA Adapter Weights
import shutil
import inspect
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig

print('=' * 80)
print('STEP 3: MERGING LORA WEIGHTS INTO QWEN2.5-1.5B')
print('=' * 80)

BASE_MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
MERGED_DIR = '/kaggle/working/qwen2.5-1.5b-rebuiltv4-merged'

print(f'[*] Loading base tokenizer: {BASE_MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)

print(f'[*] Loading base model in float16...')
device_map = 'auto' if torch.cuda.is_available() else None
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch_dtype,
    device_map=device_map,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

print(f'[*] Attaching RebuiltV4 Checkpoint-200 adapter from {adapter_dir}...')
with open(os.path.join(adapter_dir, 'adapter_config.json'), 'r') as f:
    cfg_raw = json.load(f)

sig = inspect.signature(LoraConfig.__init__)
cleaned_cfg = {k: v for k, v in cfg_raw.items() if k in sig.parameters and v is not None}
cleaned_cfg['r'] = cfg_raw.get('r', 16)
cleaned_cfg['lora_alpha'] = cfg_raw.get('lora_alpha', 32)
cleaned_cfg['target_modules'] = cfg_raw.get('target_modules', ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
lora_config = LoraConfig(**cleaned_cfg)

peft_model = PeftModel.from_pretrained(base_model, adapter_dir, config=lora_config)

print('[*] Executing model.merge_and_unload()...')
merged_model = peft_model.merge_and_unload()

print(f'[*] Saving merged deployment model to {MERGED_DIR}...')
os.makedirs(MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

# Preserve chat template
chat_tmpl_src = os.path.join(adapter_dir, 'chat_template.jinja')
if os.path.exists(chat_tmpl_src):
    shutil.copy2(chat_tmpl_src, os.path.join(MERGED_DIR, 'chat_template.jinja'))

print('[+] Merged HuggingFace weights successfully exported.')
# Free GPU memory
del peft_model
del merged_model
del base_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Cell 4: GGUF Conversion and Q4_K_M Quantization
import hashlib

print('=' * 80)
print('STEP 4: GGUF CONVERSION & Q4_K_M QUANTIZATION')
print('=' * 80)

CONVERT_SCRIPT = os.path.join(LLAMA_CPP_DIR, 'convert_hf_to_gguf.py')
FP16_GGUF = '/kaggle/working/qwen2.5-1.5b-rebuiltv4.f16.gguf'
DEPLOYMENT_DIR = '/kaggle/working/deployment'
os.makedirs(DEPLOYMENT_DIR, exist_ok=True)
FINAL_Q4_GGUF = os.path.join(DEPLOYMENT_DIR, 'qwen2.5-1.5b-rebuiltv4.Q4_K_M.gguf')

print('[*] Step 4a: Converting merged model to intermediate FP16 GGUF...')
cmd_convert = [
    sys.executable,
    CONVERT_SCRIPT,
    MERGED_DIR,
    '--outfile', FP16_GGUF,
    '--outtype', 'f16',
]
subprocess.run(cmd_convert, check=True)
fp16_size_mb = round(os.path.getsize(FP16_GGUF) / (1024 * 1024), 2)
print(f'[+] Intermediate FP16 GGUF generated: {fp16_size_mb} MB')

# Remove merged HuggingFace directory to conserve disk space
print('[*] Cleaning up intermediate HuggingFace directory...')
shutil.rmtree(MERGED_DIR, ignore_errors=True)

print('[*] Step 4b: Quantizing FP16 GGUF to Q4_K_M for 8 GB RAM deployment...')
cmd_quant = [
    quant_bin,
    FP16_GGUF,
    FINAL_Q4_GGUF,
    'Q4_K_M',
]
subprocess.run(cmd_quant, check=True)

assert os.path.exists(FINAL_Q4_GGUF), 'Quantized GGUF not found!'
q4_size_bytes = os.path.getsize(FINAL_Q4_GGUF)
q4_size_mb = round(q4_size_bytes / (1024 * 1024), 2)
print(f'[+] Final Quantized Q4_K_M GGUF generated: {q4_size_mb} MB')

# Compute SHA256
print('[*] Computing SHA256 digest...')
sha256 = hashlib.sha256()
with open(FINAL_Q4_GGUF, 'rb') as f:
    for block in iter(lambda: f.read(65536), b''):
        sha256.update(block)
sha256_digest = sha256.hexdigest()
print(f'[+] Verified SHA256: {sha256_digest}')

# Remove intermediate FP16 GGUF to leave maximum free disk space
if os.path.exists(FP16_GGUF):
    os.remove(FP16_GGUF)
print('[+] Intermediate FP16 GGUF removed.')


In [ ]:
# Cell 5: Rigorous In-Kaggle Validation and Prompt Testing
import time
import re

print('=' * 80)
print('STEP 5: RIGOROUS IN-KAGGLE VALIDATION & PROMPT TESTING')
print('=' * 80)

validation_records = []

test_prompts = [
    {
        'name': 'Benign API Traffic',
        'prompt': '<|im_start|>system\nYou are an API Security Classifier. Return JSON ONLY with key category.\n<|im_end|>\n<|im_start|>user\nClassify: GET /api/v1/health HTTP/1.1\n<|im_end|>\n<|im_start|>assistant\n',
        'expected_category': 'Benign',
    },
    {
        'name': 'SQL Injection Probe',
        'prompt': '<|im_start|>system\nYou are an API Security Classifier. Return JSON ONLY with key category.\n<|im_end|>\n<|im_start|>user\nClassify: GET /api/users?id=1\\' OR 1=1-- HTTP/1.1\n<|im_end|>\n<|im_start|>assistant\n',
        'expected_category': 'SQL Injection',
    },
    {
        'name': 'Directory Traversal Probe',
        'prompt': '<|im_start|>system\nYou are an API Security Classifier. Return JSON ONLY with key category.\n<|im_end|>\n<|im_start|>user\nClassify: GET /api/download?file=../../../../etc/passwd HTTP/1.1\n<|im_end|>\n<|im_start|>assistant\n',
        'expected_category': 'Directory Traversal',
    },
    {
        'name': 'XSS Injected Payload',
        'prompt': '<|im_start|>system\nYou are an API Security Classifier. Return JSON ONLY with key category.\n<|im_end|>\n<|im_start|>user\nClassify: POST /api/comments body={"text": "<script>alert(1)</script>"}\n<|im_end|>\n<|im_start|>assistant\n',
        'expected_category': 'XSS',
    },
    {
        'name': 'Evidence Analysis (SQL Error)',
        'prompt': '<|im_start|>system\nAnalyze HTTP response evidence. Return JSON ONLY with status and category.\n<|im_end|>\n<|im_start|>user\nTool: httpx_request. Status: 500. Body: unrecognized token near \'\'\'. SQL syntax error.\n<|im_end|>\n<|im_start|>assistant\n',
        'expected_status': 'CONFIRMED',
    },
    {
        'name': 'Structured Tool Action Selection',
        'prompt': '<|im_start|>system\nPropose next action. Return JSON ONLY with action_type and tool_name.\n<|im_end|>\n<|im_start|>user\nTurn 1/10. Available tools: httpx_request, parameter_fuzzer, jwt_inspector. Endpoints: /api/search\n<|im_end|>\n<|im_start|>assistant\n',
        'expected_action': 'execute_tool',
    },
]

for tp in test_prompts:
    t0 = time.time()
    cmd_infer = [
        cli_bin,
        '-m', FINAL_Q4_GGUF,
        '-p', tp['prompt'],
        '-n', '128',
        '-t', '4',
        '-c', '1536',
        '--temp', '0.0',
        '--log-disable',
    ]
    res = subprocess.run(cmd_infer, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    raw_out = res.stdout.strip()
    dt = round(time.time() - t0, 3)
    
    # Parse JSON
    parsed = None
    json_match = re.search(r'\{.*?\}', raw_out, re.DOTALL)
    if json_match:
        try:
            parsed = json.loads(json_match.group(0))
        except Exception:
            pass

    rec = {
        'name': tp['name'],
        'latency_seconds': dt,
        'raw_output': raw_out,
        'parsed_json': parsed,
        'valid_json': parsed is not None,
    }
    validation_records.append(rec)
    print(f'[*] Test: {tp["name"]} | Time: {dt}s | Valid JSON: {parsed is not None} | Output: {raw_out[:90]}...')

valid_json_count = sum(1 for r in validation_records if r['valid_json'])
print(f'[+] Validation Results: {valid_json_count}/{len(test_prompts)} tests produced valid structured JSON.')


In [ ]:
# Cell 6: Manifest Generation and Packaging for Download
print('=' * 80)
print('STEP 6: DEPLOYMENT MANIFEST & CHECKSUM CREATION')
print('=' * 80)

manifest = {
    'manifest_version': '1.0',
    'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'model_id': 'qwen2.5-1.5b-rebuiltv4',
    'base_model': BASE_MODEL_NAME,
    'adapter_source': adapter_dir,
    'adapter_checkpoint': 'checkpoint-200',
    'deployment_target': 'Windows 10/11 x64, 8 GB RAM, Intel i3-class CPU (Offline)',
    'quantization': 'Q4_K_M',
    'file_name': 'qwen2.5-1.5b-rebuiltv4.Q4_K_M.gguf',
    'file_size_bytes': q4_size_bytes,
    'file_size_mb': q4_size_mb,
    'sha256': sha256_digest,
    'runtime_engine': 'llama-cpp-python / llama.cpp',
    'recommended_config': {
        'n_ctx': 1536,
        'n_threads': 4,
        'n_batch': 128,
        'use_mmap': True,
        'estimated_ram_mb': 920.0,
    },
    'validation_summary': {
        'tests_executed': len(validation_records),
        'valid_structured_json': valid_json_count,
    }
}

manifest_path = os.path.join(DEPLOYMENT_DIR, 'deployment_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

sums_path = os.path.join(DEPLOYMENT_DIR, 'SHA256SUMS.txt')
with open(sums_path, 'w') as f:
    f.write(f'{sha256_digest}  qwen2.5-1.5b-rebuiltv4.Q4_K_M.gguf\n')

val_path = os.path.join(DEPLOYMENT_DIR, 'export_validation.json')
with open(val_path, 'w') as f:
    json.dump(validation_records, f, indent=2)

# Also copy artifacts to root /kaggle/working/ so they are immediately accessible
root_gguf = '/kaggle/working/qwen2.5-1.5b-rebuiltv4.Q4_K_M.gguf'
root_manifest = '/kaggle/working/deployment_manifest.json'
root_sums = '/kaggle/working/SHA256SUMS.txt'
root_val = '/kaggle/working/export_validation.json'

shutil.copy2(FINAL_Q4_GGUF, root_gguf)
shutil.copy2(manifest_path, root_manifest)
shutil.copy2(sums_path, root_sums)
shutil.copy2(val_path, root_val)

print(f'[+] Deployment Artifacts successfully placed in /kaggle/working/ :')
print(f'    - {root_gguf} ({q4_size_mb} MB)')
print(f'    - {root_manifest}')
print(f'    - {root_sums}')
print(f'    - {root_val}')
print('[+] EXPORT PIPELINE FULLY COMPLETED WITH SUCCESS.')
